In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 45.5 MB/s eta 0:00:00


In [2]:
from google.colab import files

uploaded = files.upload()

Saving Natural_Language_v.csv to Natural_Language_v.csv


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv(list(uploaded.keys())[0])

In [5]:
df.sample(2)

,document,embedding
1956,PAPER ID : 2604.21860v1\n\n Title : Transient...,[-6.61784858e-02 -1.13685578e-02 -3.05922870e-...
7124,PAPER ID : 2602.01978v1\n\n Title : SpikingGa...,[-0.08277518 -0.01079348 -0.00904339 -0.013967...


In [11]:
import numpy as np

def to_np(val):
  return np.array(val)

In [12]:
df['embedding'] = df['embedding'].apply(to_np)

In [15]:
df.loc[1,'embedding']

'[-9.21909325e-03  5.93710877e-02  7.51834537e-04  2.69827573e-03\n  1.73840597e-02  1.35339536e-02 -2.96685454e-02  4.69443249e-03\n  5.50504886e-02 -4.85106558e-02  3.14437225e-02 -2.46297810e-02\n  1.59274749e-02  2.81924792e-02  7.11837858e-02  2.41593756e-02\n  1.45889129e-02 -3.80828716e-02 -5.22352941e-03 -6.78416938e-02\n  2.44609881e-02 -3.49539183e-02  9.37767979e-03  1.20226629e-02\n -3.19534913e-02  8.25701095e-03 -5.97369820e-02 -4.41436768e-02\n -7.77168572e-02 -2.48895004e-01 -1.47560574e-02 -8.69262964e-03\n  7.22438172e-02  4.61254865e-02 -5.62162288e-02 -1.70690790e-02\n  9.61494911e-03 -1.82659160e-02  6.02002628e-02 -1.07767377e-02\n -1.82731766e-02  2.06478722e-02  9.44411661e-03  2.22772341e-02\n -5.36917802e-03 -4.78907004e-02  4.46787290e-03 -1.06897093e-02\n -8.84105265e-02  1.30949272e-02  9.43760015e-03 -3.44655029e-02\n  8.37450195e-03  7.03954175e-02  4.78494838e-02  7.72547647e-02\n  1.78357698e-02  7.15056360e-02 -3.78043652e-02  3.88901383e-02\n  1.48389

In [17]:
print(type(df.loc[0, 'embedding']))

<class 'str'>


In [18]:
import numpy as np

df['embedding'] = df['embedding'].apply(
    lambda s: np.fromstring(s.strip('[]'), sep=' ')
)

In [19]:
embeddings = np.stack(df['embedding'].values).astype(np.float32)

In [20]:
dimension = embeddings.shape[1]

In [21]:
import faiss

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

In [22]:
query = "Explain Retrieval Augmented Generation"

In [23]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [24]:
query_embedding = model.encode(
    [query],
    convert_to_numpy=True
).astype("float32")

In [25]:
k = 3

In [26]:
distances, indices = index.search(query_embedding, k)

In [28]:
for i in indices:
  print(df.loc[i,'document'])

2204    PAPER ID : 2604.22849v1\n\n  Title : RAG: Retr...
6843    PAPER ID : 2603.08819v3\n\n  Title : Beyond Re...
5159    PAPER ID : 2604.11407v2\n\n  Title : Retrieval...
Name: document, dtype: object
